# tda-witness demo

Topological Data Analysis from scratch. Point cloud in, Betti numbers out.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/avacadobanana352/tda-witness/blob/main/examples/tda_witness_demo.ipynb)

In [ ]:
!pip install -q --no-cache-dir "tda-witness[all] @ git+https://github.com/avacadobanana352/tda-witness.git"

In [ ]:
import numpy as np
from tda import analyze, persistent_homology
from tda.datasets import make_circle, make_figure_eight, make_sphere
from tda.visualization import (
    plot_complex, plot_filtration, plot_betti_summary,
    plot_persistence_diagram, plot_barcode,
)

## 1. Circle

A circle has one connected component and one loop: $\beta_0 = 1, \beta_1 = 1$.

In [ ]:
circle = make_circle(n_points=50, noise=0.05, seed=42)
result = analyze(circle, threshold=0.3, seed=42)
print("Betti numbers:", result["betti"])

In [ ]:
plot_complex(result["data"], result["landmarks"], result["graph"], result["complex"])

### Filtration

Sweep the threshold R and watch the complex grow. At low R you see many disconnected components. As R increases, edges appear and the loop forms.

In [ ]:
thresholds = np.linspace(0.05, 0.5, 25)
plot_filtration(circle, thresholds=thresholds, seed=42)

### Betti numbers vs. threshold

In [ ]:
betti_seqs = [analyze(circle, threshold=float(r), seed=42)["betti"] for r in thresholds]
plot_betti_summary(thresholds, betti_seqs)

### Persistent homology

No threshold needed — persistence sweeps them all and tracks when features are born and die.

In [ ]:
ph = persistent_homology(circle, simplex_dim=2, seed=42)

# show only the significant pairs (lifetime > 0.1)
import math
for p in ph["pairs"]:
    lt = p["death"] - p["birth"] if not math.isinf(p["death"]) else float("inf")
    if lt > 0.1:
        death = "inf" if math.isinf(p["death"]) else f'{p["death"]:.3f}'
        print(f'  H{p["dim"]}: [{p["birth"]:.3f}, {death})  lifetime={lt if math.isinf(lt) else f"{lt:.3f}"}')

In [ ]:
plot_persistence_diagram(ph["pairs"])

In [ ]:
plot_barcode(ph["pairs"], min_lifetime=0.05)

## 2. Figure eight

Two loops: $\beta_0 = 1, \beta_1 = 2$.

In [ ]:
fig8 = make_figure_eight(n_points=80, noise=0.03, seed=42)
result8 = analyze(fig8, threshold=0.25, seed=42)
print("Betti numbers:", result8["betti"])
plot_complex(result8["data"], result8["landmarks"], result8["graph"], result8["complex"])

## 3. Sphere

A sphere has one connected component and no loops: $\beta_0 = 1, \beta_1 = 0$. (The enclosed void $\beta_2 = 1$ requires `simplex_dim=3`, which is expensive for large point clouds.)

In [ ]:
sphere = make_sphere(n_points=100, noise=0.02, seed=42)
result_s = analyze(sphere, threshold=0.5, simplex_dim=2, seed=42)
print("Betti numbers:", result_s["betti"])
plot_complex(result_s["data"], result_s["landmarks"], result_s["graph"], result_s["complex"])

## 4. Bring your own data

Any `(n_points, n_dims)` numpy array works.

In [ ]:
# random point cloud
rng = np.random.default_rng(0)
my_data = rng.standard_normal((30, 2))

result_custom = analyze(my_data, threshold=0.8)
print("Betti numbers:", result_custom["betti"])
plot_complex(result_custom["data"], result_custom["landmarks"], result_custom["graph"], result_custom["complex"])

## 5. Real-world example: MNIST digits

Handwritten digits fall into three topological classes by β₁: no holes (1,2,3,5,7), one hole (0,4,6,9), two holes (8). Topology can't distinguish within a class — a 0 and a 6 look the same — but β₁ is a robust, training-free feature. See the full example:

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/avacadobanana352/tda-witness/blob/main/examples/mnist_topology.ipynb)